# Phase 7 - Dataset Preparation for QLoRA Fine-Tuning

Tokenize real train/val/test CSVs using Qwen2.5-3B tokenizer.
Loss computed **ONLY on the assistant JSON** response — prompt is masked with `-100`.

## Prerequisites
1. **Runtime → Change runtime type → T4 GPU**
2. Google Drive mounted with repo present
3. `data/splits/{train,validation,test}.csv` exist (Phase 5 done)
4. `models/base/Qwen2.5-3B/` downloaded (Phase 6 done)

**Run cells in order. Do not skip.**

In [ ]:
# Cell 1 - Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
# Cell 2 - Set REPO_DIR
# Adjust to where your repo lives on Drive:
#   /content/drive/MyDrive/telecom-support-ticket-triage
#   /content/drive/MyDrive/GenAI-Projects/telecom-support-ticket-triage
import os
REPO_DIR = '/content/drive/MyDrive/telecom-support-ticket-triage'
assert os.path.isdir(REPO_DIR), 'REPO_DIR not found: ' + REPO_DIR
os.chdir(REPO_DIR)
print('Working dir:', os.getcwd())
print('Repo contents:', os.listdir('.'))

In [ ]:
# Cell 3 - Git pull latest changes
!git pull
!git log --oneline -5

In [ ]:
# Cell 4 - Install training dependencies
# triton==2.3.0 is pinned to fix known Colab T4 incompatibility
!pip install -q -r training/requirements-colab.txt
!pip install -q triton==2.3.0
print('Installation complete.')

In [ ]:
# Cell 5 - Hardware check
import torch
import transformers, peft, trl, bitsandbytes, datasets, accelerate
print('PyTorch:     ', torch.__version__)
print('Transformers:', transformers.__version__)
print('PEFT:        ', peft.__version__)
print('TRL:         ', trl.__version__)
print('bitsandbytes:', bitsandbytes.__version__)
print('datasets:    ', datasets.__version__)
print('accelerate:  ', accelerate.__version__)
print()
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU: ', torch.cuda.get_device_name(0))
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print('VRAM:', round(vram, 1), 'GB')
else:
    print('WARNING: No GPU. Switch to T4 before Phase 8 training.')

In [ ]:
# Cell 6 - Verify prerequisite files exist
from pathlib import Path
import csv
missing = []
for split in ['train.csv', 'validation.csv', 'test.csv']:
    p = Path('data/splits') / split
    if not p.exists():
        missing.append(str(p))
        print('MISSING:', p)
    else:
        with open(p, encoding='utf-8') as f:
            rows = list(csv.DictReader(f))
        print(split + ':', len(rows), 'rows')
model_dir = Path('models/base/Qwen2.5-3B')
if not model_dir.exists():
    missing.append(str(model_dir))
    print('MISSING: models/base/Qwen2.5-3B - download Phase 6 base model first')
else:
    print('Base model:', len(list(model_dir.iterdir())), 'files found')
if not missing:
    print('All prerequisites OK.')

In [ ]:
# Cell 7 - Run prepare_dataset.py (REAL run - no --self-test flag)
# Tokenizes all 3 splits, saves to training/prepared/{train,validation,test}/
# Runtime: ~2-5 minutes on CPU
!python training/prepare_dataset.py \
    --model-path models/base/Qwen2.5-3B \
    --max-length 512

In [ ]:
# Cell 8 - Verify output: counts + label masking check
from datasets import load_from_disk
from pathlib import Path
all_ok = True
for name in ['train', 'validation', 'test']:
    path = Path('training/prepared') / name
    if not path.exists():
        print('ERROR: not found -', path)
        all_ok = False
        continue
    ds = load_from_disk(str(path))
    ex = ds[0]
    n_masked = sum(1 for l in ex['labels'] if l == -100)
    n_loss   = len(ex['labels']) - n_masked
    ok = n_loss > 0 and n_masked > 0
    if not ok:
        all_ok = False
    status = 'OK' if ok else 'FAIL'
    print(name + ': ' + str(len(ds)) + ' examples | masked=' + str(n_masked) + ' loss_tokens=' + str(n_loss) + ' | ' + status)
print()
print('Phase 7 verification:', 'PASSED' if all_ok else 'FAILED - fix errors above')

In [ ]:
# Cell 9 - Decode first training example for human review
from datasets import load_from_disk
from transformers import AutoTokenizer
import json
tok = AutoTokenizer.from_pretrained('models/base/Qwen2.5-3B')
ds  = load_from_disk('training/prepared/train')
ex  = ds[0]
full_text  = tok.decode(ex['input_ids'], skip_special_tokens=False)
label_ids  = [t for t in ex['labels'] if t != -100]
label_text = tok.decode(label_ids, skip_special_tokens=True)
print('=== FULL INPUT (first 1200 chars) ===')
print(full_text[:1200])
print('\n=== LABEL (what model learns to predict) ===')
print(label_text)
try:
    parsed = json.loads(label_text.strip())
    assert all(k in parsed for k in ['category', 'priority', 'department'])
    print('\nLabel JSON VALID:', parsed)
except Exception as e:
    print('\nWARNING: label not valid JSON:', e)

In [ ]:
# Cell 10 - Token length distribution (validates max_length=512 choice)
from datasets import load_from_disk
import numpy as np
ds = load_from_disk('training/prepared/train')
lens = [len(ex['input_ids']) for ex in ds]
print('min:', min(lens), ' max:', max(lens), ' mean:', round(float(np.mean(lens)), 1))
print('p95:', round(float(np.percentile(lens, 95)), 1), ' p99:', round(float(np.percentile(lens, 99)), 1))
trunc = sum(1 for l in lens if l == 512)
pct   = trunc / len(lens) * 100
print('Truncated at max_length=512:', trunc, '(' + str(round(pct, 1)) + '%)')
if pct > 5:
    print('RECOMMENDATION: >5% truncated.')
    print('Re-run Cell 7 with --max-length 768, then update Phase 8 train command too.')
else:
    print('max_length=512 is adequate for this dataset.')

In [ ]:
# Cell 11 - Save metadata JSON + git commit
# Binary HF arrow files are NOT committed (too large).
# meta.json is small and tracks stats for future sessions.
import json
import numpy as np
from datasets import load_from_disk
from pathlib import Path
meta = {}
for name in ['train', 'validation', 'test']:
    ds   = load_from_disk('training/prepared/' + name)
    lens = [len(ex['input_ids']) for ex in ds]
    meta[name] = {
        'num_examples': len(ds),
        'columns': ds.column_names,
        'token_length_mean': round(float(np.mean(lens)), 1),
        'token_length_max': int(max(lens)),
        'token_length_p95': round(float(np.percentile(lens, 95)), 1),
        'truncated_at_512': int(sum(1 for l in lens if l == 512)),
    }
p = Path('training/prepared/meta.json')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text(json.dumps(meta, indent=2), encoding='utf-8')
print(json.dumps(meta, indent=2))
!git add training/prepared/meta.json
!git commit -m 'Phase 7 complete: prepared dataset metadata'
!git push
print('\nPHASE 7 COMPLETE')
print('Next: open notebooks/phase8_train.ipynb and run training.')